In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

In [2]:
!pip install transformers==4.45.2 sentence-transformers==3.1.1
#!pip install kagglehub

In [3]:
print("test")

test


## HME100K Datensatz laden

In [4]:
import os
path = "hme100k_plus_matrices/"
#print(path.endswith(".zip"))
print(os.listdir(path))

['val.txt', 'train.txt', '.ipynb_checkpoints', 'test.txt', 'images']


## Datensatz aufteilen

In [5]:
from pathlib import Path

def read_lines(txt_path):
    return [ln for ln in Path(txt_path).read_text(encoding="utf-8").splitlines()if ln.strip()]

train_lines = read_lines("hme100k_plus_matrices/train.txt")
val_lines   = read_lines("hme100k_plus_matrices/val.txt")
test_lines  = read_lines("hme100k_plus_matrices/test.txt")

print(len(train_lines), len(val_lines), len(test_lines))


79283 9916 9906


In [6]:
print(train_lines[24])

images/64422.png	= \frac { 1 5 } { 1 7 }


## Daten Normalisieren

In [7]:
import re

def norm_line(line):
    separator = line.split("\t")
    image_path = separator[0].strip()
    label      = separator[1].strip()
    label = re.sub(r'\s+', '', label)  # Alle Whitespaces entfernen
    return image_path, label

In [8]:
from PIL import Image
def load_hme_data(lines, base_path="hme100k_plus_matrices/"):
    data = []
    for line in lines:
        image_path, label = norm_line(line)
        full_path = os.path.join(base_path, image_path)
        try:
            img = Image.open(full_path).convert("RGB")
            data.append((img, label))
        except FileNotFoundError:
            print(f"Bild nicht gefunden: {full_path}")
    return data

In [9]:
hme_train = load_hme_data(train_lines)
hme_val   = load_hme_data(val_lines)
hme_test  = load_hme_data(test_lines)
print(f"HME geladen – Train: {len(hme_train)}, Val: {len(hme_val)}, Test: {len(hme_test)}")

HME geladen – Train: 79283, Val: 9916, Test: 9906


In [10]:
image, label = hme_train[24]
print(label)

=\frac{15}{17}


In [11]:
from datasets import load_dataset
ds = load_dataset("deepcopy/MathWriting-Human")
mw_train = [(s["image"], s["latex"]) for s in ds["train"]]
mw_val   = [(s["image"], s["latex"]) for s in ds["val"]]
mw_test  = [(s["image"], s["latex"]) for s in ds["test"]]
print(f"MathWriting – Train: {len(mw_train)}, Val: {len(mw_val)}, Test: {len(mw_test)}")

MathWriting – Train: 229864, Val: 15674, Test: 7644


In [12]:
import random
combined_train = mw_train + hme_train
combined_val   = mw_val   + hme_val
combined_test  = mw_test  + hme_test

random.shuffle(combined_train)  # Nur Train mischen

print(f"Kombiniert – Train: {len(combined_train)}, Val: {len(combined_val)}, Test: {len(combined_test)}")

Kombiniert – Train: 309147, Val: 25590, Test: 17550


## Dataset-Klasse für den HME-Datensatz:

- Bild und die zugehörige LaTeX wird geladen
- Verarbeitet das Bild mit dem Processor zu pixel_values und tokenisiert den LaTeX-Text zu input_ids
- Ersetzt die PAD-Tokens durch -100, damit sie beim Loss ignoriert werden
- Gibt ein Trainingssample pixel_values und labels zurück

In [13]:
from torch.utils.data import Dataset
from PIL import Image
import torch

class MathWritingDataset(Dataset):
    def __init__(self, data, processor, max_target_length=128):
        self.data = data
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.data) # Wird für die Batch-Verarbeitung gebraucht (DataLoader)

    # Zugriff auf ein einzelnes Sample:
    def __getitem__(self, idx):
        image, latex = self.data[idx]
        image = image.convert("RGB")

        pixel_values = self.processor(image, return_tensors="pt").pixel_values #https://stackoverflow.com/questions/75891072/valueerror-unable-to-infer-channel-dimension-format
        labels = self.processor.tokenizer(text_target=latex ,padding="max_length",truncation=True,max_length=self.max_target_length).input_ids  # input_ids: Liste der Token-Zahlen
        # Padding Tokens werden hier ignoriert: <PAD> Tokens werden bei zu kurzen Texten als Auffüll-Zeichen genutzt, der Cross-Entropy Loss soll diese ignorieren (geschieht durch setzen von -100)
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels] 

        encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)} # encoding hat: pixel_values (Tensor des Bildes), labels (Tensor der Token-IDs des Textes)
        return encoding

In [14]:
from transformers import TrOCRProcessor
from torch.utils.data import Subset
import random

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-large-stage1')

train_dataset = MathWritingDataset(combined_train,processor)
val_dataset = MathWritingDataset(combined_val, processor)
test_dataset = MathWritingDataset(combined_test, processor)

## Modell laden

In [15]:
from transformers import VisionEncoderDecoderModel
import torch
import inspect

#https://huggingface.co/fhswf/TrOCR_Math_handwritten

model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-large-stage1")
#model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-large-handwritten")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(inspect.getsource(VisionEncoderDecoderModel.forward))

VisionEncoderDecoderModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-stage1 and are newly initialized: ['enco

    @add_start_docstrings_to_model_forward(VISION_ENCODER_DECODER_INPUTS_DOCSTRING)
    @replace_return_docstrings(output_type=Seq2SeqLMOutput, config_class=_CONFIG_FOR_DOC)
    def forward(
        self,
        pixel_values: Optional[torch.FloatTensor] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.BoolTensor] = None,
        encoder_outputs: Optional[Tuple[torch.FloatTensor]] = None,
        past_key_values: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
        decoder_inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        **kwargs,
    ) -> Union[Tuple[torch.FloatTensor], Seq2SeqLMOutput]:
        r"""
        Returns:

        Examples:

        ```python
        >>> f

## Konfigurationen von den Special-Tokens und Beam-Search für die Textgenerierung

In [16]:
# interessante Quelle (zum nachrecherchieren): https://huggingface.co/docs/transformers/main/model_doc/trocr   und https://huggingface.co/docs/transformers/main_classes/text_generation
# set special tokens used for creating the decoder_input_ids from the labels
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id #Normalerweise ist Starttoken <BOS> hier aber <CLS>
model.config.pad_token_id = processor.tokenizer.pad_token_id 
# make sure vocab size is set correctly
model.config.vocab_size = model.config.decoder.vocab_size


# set beam search parameters
model.config.eos_token_id = processor.tokenizer.sep_token_id

In [17]:
from transformers import GenerationConfig

model.generation_config = GenerationConfig(
    decoder_start_token_id=processor.tokenizer.cls_token_id,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.sep_token_id,
    max_length=128,
    num_beams=4,
    early_stopping=True,
    length_penalty=1.0
)



## Laden der CER-Metrik und Editdistance:

In [18]:
!pip install editdistance

In [19]:
from evaluate import load
#import editdistance
cer = load("cer")


In [20]:
import editdistance
def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    
    cer_score = cer.compute(predictions=pred_str, references=label_str)
    
    # Edit Distance Schwellenwert-Metriken
    edit_distances = [
        editdistance.eval(p, r)
        for p, r in zip(pred_str, label_str)
    ]
    n = len(edit_distances)
    exact_match      = sum(d == 0 for d in edit_distances) / n
    one_error_match  = sum(d <= 1 for d in edit_distances) / n
    two_error_match  = sum(d <= 2 for d in edit_distances) / n
    
    return {
        "cer": cer_score,
        "exact_match":     exact_match,      
        "one_error_match": one_error_match,  
        "two_error_match": two_error_match, 
    }

## Trainingsparameter einstellen:

In [21]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
# https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer#transformers.Seq2SeqTrainingArguments
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    evaluation_strategy="epoch",
    per_device_train_batch_size=32, # Batchgröße pro CPU/GPU
    per_device_eval_batch_size=32,
    bf16=True,  # Teile des Trainings laufen mit 16 Bit (spart Speicher und beschleunigt das Training)
    output_dir="./model-mathwriting_and_hme_large_stage_abgabe", #Verzeichnis in dem Modelle etc. gespeichert werden_
    logging_steps=100,
    save_strategy="epoch",
    save_steps=3000,
    seed=42,
    num_train_epochs=4,
    gradient_accumulation_steps=2,
    warmup_ratio= 0.1,
    learning_rate=3.5e-5
)

/opt/conda/lib/python3.11/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [29]:
print(training_args.generation_max_length, training_args.warmup_ratio)

None 0.1


In [30]:
from transformers import default_data_collator
torch.cuda.empty_cache()
# https://huggingface.co/docs/transformers/main_classes/trainer

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.tokenizer,
    args=training_args, # Sind oben definiert
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    #eval_dataset=small_eval_dataset,
    data_collator=default_data_collator,  # collator: Erstellt Batches aus den Elementen der Trainings oder Validierungsdaten
)

trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sandzo (sandzo-fh-swf) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Cer,Exact Match,One Error Match,Two Error Match
0,0.148200,0.180132,0.090526,0.594490,0.713990,0.760414
2,0.053600,0.112282,0.056908,0.705080,0.804103,0.838882
3,0.028700,0.108101,0.052374,0.721180,0.817429,0.849433


TrainOutput(global_step=19320, training_loss=0.17733168892300152, metrics={'train_runtime': 44010.4501, 'train_samples_per_second': 28.098, 'train_steps_per_second': 0.439, 'total_flos': 1.8302731377228487e+21, 'train_loss': 0.17733168892300152, 'epoch': 3.999585964185902})

## Evaluieren

In [31]:
from transformers import default_data_collator
torch.cuda.empty_cache()
# https://huggingface.co/docs/transformers/main_classes/trainer

trained_model = VisionEncoderDecoderModel.from_pretrained('model-mathwriting_and_hme_large_stage_abgabe/checkpoint-19320/').to(device)

t_trainer = Seq2SeqTrainer(
    model=trained_model,
    tokenizer=processor.tokenizer,
    args=training_args, # Sind oben definiert
    compute_metrics=compute_metrics,
    #train_dataset=train_dataset,
    #eval_dataset=val_dataset,
    eval_dataset=test_dataset,
    #eval_dataset=small_eval_dataset,
    data_collator=default_data_collator  # collator: Erstellt Batches aus den Elementen der Trainings oder Validierungsdaten
)

In [32]:
test_res = t_trainer.evaluate()

print(test_res)

{'eval_loss': 0.12119897454977036, 'eval_model_preparation_time': 0.0251, 'eval_cer': 0.057695507026163724, 'eval_exact_match': 0.6822222222222222, 'eval_one_error_match': 0.7891168091168091, 'eval_two_error_match': 0.8301994301994302, 'eval_runtime': 1881.7452, 'eval_samples_per_second': 9.326, 'eval_steps_per_second': 0.292}


## Nur Mathwriting: 

In [22]:
from transformers import default_data_collator
torch.cuda.empty_cache()
# https://huggingface.co/docs/transformers/main_classes/trainer

trained_model = VisionEncoderDecoderModel.from_pretrained('model-mathwriting_large_stage_abgabe/checkpoint-14368/').to(device)

t_trainer = Seq2SeqTrainer(
    model=trained_model,
    tokenizer=processor.tokenizer,
    args=training_args, # Sind oben definiert
    compute_metrics=compute_metrics,
    #train_dataset=train_dataset,
    #eval_dataset=val_dataset,
    eval_dataset=test_dataset,
    #eval_dataset=small_eval_dataset,
    data_collator=default_data_collator  # collator: Erstellt Batches aus den Elementen der Trainings oder Validierungsdaten
)

In [23]:
test_res = t_trainer.evaluate()

print(test_res)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sandzo (sandzo-fh-swf) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'eval_loss': 0.6963634490966797, 'eval_model_preparation_time': 0.0127, 'eval_cer': 0.26972704989851715, 'eval_exact_match': 0.3847863247863248, 'eval_one_error_match': 0.4863817663817664, 'eval_two_error_match': 0.5376068376068376, 'eval_runtime': 1979.3736, 'eval_samples_per_second': 8.866, 'eval_steps_per_second': 0.277}


## Untrainiertes Modell

In [33]:
processor_base = TrOCRProcessor.from_pretrained('microsoft/trocr-large-stage1')
base_model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-large-stage1').to(device)

# Gleiche Configs wie beim Training
base_model.config.decoder_start_token_id = processor_base.tokenizer.cls_token_id
base_model.config.pad_token_id = processor_base.tokenizer.pad_token_id
base_model.config.vocab_size = base_model.config.decoder.vocab_size
base_model.config.eos_token_id = processor_base.tokenizer.sep_token_id

from transformers import GenerationConfig
base_model.generation_config = GenerationConfig(
    decoder_start_token_id=processor_base.tokenizer.cls_token_id,
    pad_token_id=processor_base.tokenizer.pad_token_id,
    eos_token_id=processor_base.tokenizer.sep_token_id,
    max_length=128,
    num_beams=4,
    early_stopping=True,
    length_penalty=1.0
)

base_trainer = Seq2SeqTrainer(
    model=base_model,
    tokenizer=processor_base.tokenizer,
    args=training_args,
    compute_metrics=compute_metrics,
    eval_dataset=test_dataset,
    data_collator=default_data_collator
)

base_res = base_trainer.evaluate()
print(base_res)

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-stage1 and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'eval_loss': 10.960025787353516, 'eval_model_preparation_time': 0.0078, 'eval_cer': 1.0142608387032375, 'eval_exact_match': 0.0, 'eval_one_error_match': 0.0023361823361823363, 'eval_two_error_match': 0.0105982905982906, 'eval_runtime': 1321.2586, 'eval_samples_per_second': 13.283, 'eval_steps_per_second': 0.416}
